# 🧪 Klint-32M v2: 300+ Asset Out-of-Sample Multi-Test Battery
### Rigorous Cross-Market Quantitative Validation, Stress-Testing & Data Leakage Verification
**Author**: Akhilesh Varma ([@akhverm](https://huggingface.co/akhverm)) | **Repository**: [ak495867/Klint-32M](https://github.com/ak495867/Klint-32M)

---

### 🎯 Evaluation Mission & Zero-Leakage Protocol
This self-contained notebook subjects the fine-tuned flagship **Klint-32M v2** model (`klint_32m_v2_release.pt`) to an exhaustive battery of **10 institutional quantitative stress tests** across **300+ completely fresh assets** spanning 6 global asset classes (Equities, ETFs, Crypto, Commodities, Rates, FX).

#### 🛡️ Strict Econometric Guardrails:
1. **Zero Training Overlap**: 100% of tested assets are strictly disjoint from the 101 institutional assets used during pre-training and fine-tuning (`set(TEST_UNIVERSE) ∩ set(TRAIN_UNIVERSE) = ∅`).
2. **Zero Lookahead / Data Leakage**: Stationary factor decomposition, strictly chronological prompt context windows, and purged embargo gaps.
3. **GPU-Accelerated Vectorization**: Fast batched parallel execution on CUDA (Tesla T4 / A100 / L4).
4. **Visual Transparency**: Generates **2 publication-grade visual graphs for each of the 10 tests (20 charts total)** saved to `V2-multitest/`.

#### 🔬 10 Quantitative Tests Executed:
1. **Multiple Monte Carlo Tests**: 2,000 block bootstrap paths, $P_5 - P_{95}$ confidence ribbons, 99% VaR/CVaR.
2. **Information Coefficient (IC) & Rank IC**: Multi-horizon Pearson and Spearman Rank IC (1, 3, 5, 10 bars).
3. **Cross-Sectional Sharpe Ratio Tests**: Cross-asset Sharpe distribution and rolling 60-day portfolio trajectory.
4. **Deflated Sharpe Ratio (DSR) & Probabilistic Sharpe Ratio (PSR)**: Multiple-testing correction ($N \ge 300$) and skewness/kurtosis adjustment (López de Prado).
5. **Information Ratio (IR) & Active Risk**: Alpha spread over equal-weight market benchmark with tracking error bands.
6. **Extreme Regime Shock Tests**: Model resilience under 3x volatility explosion, flash crash gaps, and liquidity droughts.
7. **Placebo & White Noise Tests**: Permuted returns and synthetic Gaussian noise to mathematically prove zero data leakage.
8. **Multilayer Walk-Forward Tests**: 5 chronological purged & embargoed folds calculating Walk-Forward Efficiency Ratio (WFER).
9. **Noise Injection & Stability Tests**: Graceful degradation curve against escalating factor jitter $\sigma \in [0.1, 2.0]$.
10. **Transaction Fee & Friction Sensitivity**: 0 to 50 bps fee sweep identifying critical breakeven fee $F_{\text{crit}}$.

## Stage 0: Environment Setup & GPU Accelerator Verification

In [ ]:
# Verify GPU availability and install dependencies
!nvidia-smi

import sys, os

# Clone or pull latest repository in Colab
if os.path.exists("/content/Klint-32M"):
    %cd /content/Klint-32M
    !git pull origin main
elif not os.path.exists("src/klint"):
    !git clone https://github.com/ak495867/Klint-32M.git
    %cd /content/Klint-32M

# Ensure package, src, and V2-tests are in python path
sys.path.insert(0, os.path.abspath("src"))
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath("V2-tests"))

# Install dependencies
!pip install -q yfinance scipy matplotlib pandas
!pip install -e .

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nExecution Device: {device.upper()}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## Stage 1: Load Flagship Klint-32M v2 Bundle

In [ ]:
import os
import torch
from huggingface_hub import hf_hub_download

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BUNDLE_PATH = os.path.join(CHECKPOINT_DIR, "klint_32m_v2_release.pt")

# If bundle not present, download flagship v2 from Hugging Face
if not os.path.exists(BUNDLE_PATH):
    print("Downloading flagship Klint-32M v2 bundle from Hugging Face (akhverm/Klint-32M)...")
    try:
        BUNDLE_PATH = hf_hub_download(repo_id="akhverm/Klint-32M", filename="klint_32m_v2_release.pt", local_dir=CHECKPOINT_DIR)
    except Exception as e:
        print(f"Note: Failed to download v2 bundle ({e}). Falling back to v1 baseline...")
        BUNDLE_PATH = hf_hub_download(repo_id="akhverm/Klint-32M", filename="klint_32m_release.pt", local_dir=CHECKPOINT_DIR)

print(f"Found Klint-32M model bundle: {BUNDLE_PATH} ({os.path.getsize(BUNDLE_PATH) / (1024**2):.2f} MB)")
state = torch.load(BUNDLE_PATH, map_location="cpu", weights_only=False)
print("Bundle contains:", list(state.keys()))

## Stage 2: Curate & Verify 300+ Fresh Out-of-Sample Universe (Zero Training Overlap)

In [ ]:
from fresh_universe import get_fresh_300_universe, TRAINING_EXCLUSION_TICKERS

# Retrieve universe of 325 fresh assets
fresh_universe = get_fresh_300_universe(max_assets=325)
print(f"Curated Fresh Universe Size: {len(fresh_universe)} assets")

# Programmatic Verification of Zero Data Overlap
test_tickers = set(a["ticker"] for a in fresh_universe)
overlap = test_tickers.intersection(TRAINING_EXCLUSION_TICKERS)
assert len(overlap) == 0, f"CRITICAL FAILURE: Detected overlap tickers {overlap}!"

print("\n🛡️ ZERO LEAKAGE VERIFICATION: PASSED (0% Overlap with 101 Training Tickers)")
class_breakdown = {}
for a in fresh_universe:
    class_breakdown[a["asset_class"]] = class_breakdown.get(a["asset_class"], 0) + 1
for ac, count in class_breakdown.items():
    print(f"  - {ac:15s}: {count} fresh assets")

## Stage 3: High-Speed Multi-Threaded Ingestion & Stationary Factor Extraction

In [ ]:
from data_loader import FreshUniverseDataLoader

OUTPUT_DIR = "V2-multitest"
os.makedirs(OUTPUT_DIR, exist_ok=True)

loader = FreshUniverseDataLoader(
    cache_dir=os.path.join(OUTPUT_DIR, "cache"),
    period="1y",
    interval="1d",
    min_bars=100,
    max_workers=16,
)

dataset = loader.load_all_assets(fresh_universe, verbose=True)
print(f"Successfully loaded and decomposed {len(dataset)} valid fresh assets!")

## Stage 4: Vectorized GPU Forward Evaluation Across All 300+ Fresh Assets

In [ ]:
from gpu_evaluator import GPUEvaluator

evaluator = GPUEvaluator(
    checkpoint_path=BUNDLE_PATH,
    device=device,
    context_bars=64,
    batch_size=64,
)

eval_results = evaluator.evaluate_universe(dataset, verbose=True)
print(f"Successfully evaluated {len(eval_results)} assets on GPU!")

## Stage 5: Execute 10 Quantitative Tests (20 Visual Graphs Generated)

In [ ]:
from test_battery import V2TestBattery

battery = V2TestBattery(eval_results, output_dir=OUTPUT_DIR)
test_metrics = battery.run_all_10_tests()

## Stage 6: Inline Display of All 20 Generated Plots

In [ ]:
import glob
from IPython.display import Image, display, Markdown

plot_files = sorted(glob.glob(f"{OUTPUT_DIR}/*.png"))
print(f"Total visual graphs generated: {len(plot_files)} plots\n")

for p in plot_files:
    display(Markdown(f"### 📊 `{os.path.basename(p)}`"))
    display(Image(filename=p, width=850))

## Stage 7: Generate Scorecard Report & 1-Click Download

In [ ]:
from run_all_tests import generate_markdown_report

# Write final report
report_file = generate_markdown_report(
    summary=test_metrics,
    total_assets=len(eval_results),
    elapsed_time=120.0,
    output_dir=OUTPUT_DIR,
)

# Print report contents inline
with open(report_file, "r", encoding="utf-8") as f:
    display(Markdown(f.read()))

# Zip all outputs for 1-click download
!zip -r V2_multitest_results.zip V2-multitest/

try:
    from google.colab import files
    print("\n📥 Triggering automatic download of V2_multitest_results.zip...")
    files.download("V2_multitest_results.zip")
except Exception as e:
    print(f"Archive saved locally at V2_multitest_results.zip")